# 面试问题：图像分类工程如何从数据张量、模型训练走到可靠评估？

可以直接复述的回答是：首先固定图像的 HWC/CHW、dtype、取值范围和标签映射，并在训练/验证使用同一预处理。其次用简单 baseline 检查任务是否真的需要空间模型。卷积层通过局部共享核提取方向与纹理，训练时必须实际观察 loss、梯度和逐样本错误。原始 `uint8` 若不除以 255，会把 logits 与梯度放大几个数量级，混合精度时尤其危险。验证还应按来源分组避免增强副本泄漏。下面用十二张 8×8 工业图手写卷积 forward 和训练循环。

## 真实案例：区分竖直裂纹与水平划痕

十二张单通道小图包含不同位置的两像素宽竖线或横线，并带一个弱噪声点。前八张训练，后四张验证；程序图用于解释工程流程，不代表真实质检精度。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量与自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(1010)  # 固定模型参数初始化
class_names = ["竖直裂纹", "水平划痕"]  # 定义两个视觉类别
images_uint8 = torch.zeros(12, 1, 8, 8, dtype=torch.uint8)  # 初始化十二张 CHW 灰度图
labels = torch.tensor([0] * 6 + [1] * 6, dtype=torch.long)  # 前六张竖线后六张横线
for index in range(6):  # 生成六张不同位置竖直裂纹
    column = 1 + index % 5  # 选择竖线起始列
    images_uint8[index, 0, :, column:column + 2] = 255  # 绘制两像素宽竖线
    images_uint8[index, 0, index % 8, (column + 3) % 8] = 80  # 添加一个弱噪声像素
for index in range(6):  # 生成六张不同位置水平划痕
    row = 1 + index % 5  # 选择横线起始行
    images_uint8[6 + index, 0, row:row + 2, :] = 255  # 绘制两像素高横线
    images_uint8[6 + index, 0, (row + 3) % 8, index % 8] = 80  # 添加一个弱噪声像素
train_indices = torch.tensor([0, 1, 2, 3, 6, 7, 8, 9])  # 选择八张不同位置图作为训练集
validation_indices = torch.tensor([4, 5, 10, 11])  # 保留四张未见位置图作为验证集
print("输入预览：id | split | label | shape | min/max")  # 输出十二图元数据表头
for index in range(12):  # 遍历全部工业小图
    split = "train" if index in train_indices.tolist() else "valid"  # 标记当前图所属集合
    print(f"IMG-{index + 1:02d} | {split:5} | {class_names[int(labels[index])]} | {tuple(images_uint8[index].shape)} | {int(images_uint8[index].min())}/{int(images_uint8[index].max())}")  # 展示张量契约
print("竖直裂纹 ASCII：", [["#" if pixel > 128 else "." for pixel in row] for row in images_uint8[0, 0]])  # 可视化首张真实小图

输入预览：id | split | label | shape | min/max
IMG-01 | train | 竖直裂纹 | (1, 8, 8) | 0/255
IMG-02 | train | 竖直裂纹 | (1, 8, 8) | 0/255
IMG-03 | train | 竖直裂纹 | (1, 8, 8) | 0/255
IMG-04 | train | 竖直裂纹 | (1, 8, 8) | 0/255
IMG-05 | valid | 竖直裂纹 | (1, 8, 8) | 0/255
IMG-06 | valid | 竖直裂纹 | (1, 8, 8) | 0/255
IMG-07 | train | 水平划痕 | (1, 8, 8) | 0/255
IMG-08 | train | 水平划痕 | (1, 8, 8) | 0/255
IMG-09 | train | 水平划痕 | (1, 8, 8) | 0/255
IMG-10 | train | 水平划痕 | (1, 8, 8) | 0/255
IMG-11 | valid | 水平划痕 | (1, 8, 8) | 0/255
IMG-12 | valid | 水平划痕 | (1, 8, 8) | 0/255
竖直裂纹 ASCII： [['.', '#', '#', '.', '.', '.', '.', '.'], ['.', '#', '#', '.', '.', '.', '.', '.'], ['.', '#', '#', '.', '.', '.', '.', '.'], ['.', '#', '#', '.', '.', '.', '.', '.'], ['.', '#', '#', '.', '.', '.', '.', '.'], ['.', '#', '#', '.', '.', '.', '.', '.'], ['.', '#', '#', '.', '.', '.', '.', '.'], ['.', '#', '#', '.', '.', '.', '.', '.']]


## Baseline / 基线：只用平均亮度分类

竖线和横线拥有相同亮像素数量，平均亮度丢失方向信息。阈值或单一多数类只能达到约 50%。

In [2]:
images = images_uint8.float() / 255.0  # 将 uint8 图像规范化到零到一浮点范围
mean_intensity = images.mean(dim=(1, 2, 3))  # 计算每张图不含空间信息的平均亮度
baseline_prediction = (mean_intensity > 0.5).long()  # 用固定亮度阈值形成不含空间信息的基线类别
baseline_accuracy = float((baseline_prediction == labels).float().mean())  # 计算十二图基线准确率
print("id | label | mean_intensity | baseline_pred")  # 输出亮度基线表头
for index in range(12):  # 遍历十二张图
    print(f"IMG-{index + 1:02d} | {class_names[int(labels[index])]} | {mean_intensity[index]:.4f} | {class_names[int(baseline_prediction[index])]}")  # 展示空间丢失后的错误
print(f"平均亮度 baseline accuracy={baseline_accuracy:.1%}")  # 汇总同数据基线

id | label | mean_intensity | baseline_pred
IMG-01 | 竖直裂纹 | 0.2549 | 竖直裂纹
IMG-02 | 竖直裂纹 | 0.2549 | 竖直裂纹
IMG-03 | 竖直裂纹 | 0.2549 | 竖直裂纹
IMG-04 | 竖直裂纹 | 0.2549 | 竖直裂纹
IMG-05 | 竖直裂纹 | 0.2549 | 竖直裂纹
IMG-06 | 竖直裂纹 | 0.2549 | 竖直裂纹
IMG-07 | 水平划痕 | 0.2549 | 竖直裂纹
IMG-08 | 水平划痕 | 0.2549 | 竖直裂纹
IMG-09 | 水平划痕 | 0.2549 | 竖直裂纹
IMG-10 | 水平划痕 | 0.2549 | 竖直裂纹
IMG-11 | 水平划痕 | 0.2549 | 竖直裂纹
IMG-12 | 水平划痕 | 0.2549 | 竖直裂纹
平均亮度 baseline accuracy=50.0%


## 核心实现：手写局部卷积、ReLU 与全局最大池化

不调用现成 CNN 架构；forward 用滑动 `3×3` patch 与四个可训练核相乘，再通过最大池化和线性分类。

In [3]:
class TinyManualCNN(torch.nn.Module):  # 定义手写卷积图像分类器
    def __init__(self):  # 初始化卷积核与分类头
        super().__init__()  # 初始化 PyTorch 模块基类
        vertical = torch.tensor([[0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0]])  # 定义竖线先验核
        horizontal = vertical.T  # 定义横线先验核
        diagonal = torch.eye(3)  # 定义对角纹理核
        blob = torch.ones(3, 3) / 3.0  # 定义局部亮度核
        kernels = torch.stack([vertical, horizontal, diagonal, blob]).unsqueeze(1) * 0.20  # 形成四个单通道初始卷积核
        self.kernels = torch.nn.Parameter(kernels)  # 注册可训练三乘三卷积核
        self.classifier = torch.nn.Parameter(torch.randn(4, 2) * 0.10)  # 注册四特征到两类权重
        self.bias = torch.nn.Parameter(torch.zeros(2))  # 注册两类偏置
    def forward(self, batch):  # 定义滑窗卷积和分类前向
        row_outputs = []  # 收集六行卷积响应
        for row in range(6):  # 遍历有效卷积纵向位置
            column_outputs = []  # 收集当前行六列响应
            for column in range(6):  # 遍历有效卷积横向位置
                patch = batch[:, :, row:row + 3, column:column + 3]  # 提取批量三乘三局部 patch
                response = (patch[:, None, :, :, :] * self.kernels[None, :, :, :, :]).sum(dim=(2, 3, 4))  # 与四个共享核逐元素相乘求和
                column_outputs.append(response)  # 保存当前空间位置四通道响应
            row_outputs.append(torch.stack(column_outputs, dim=2))  # 拼接当前行六列响应
        convolution = torch.stack(row_outputs, dim=2)  # 形成批量乘四乘六乘六卷积特征图
        activated = torch.relu(convolution)  # 对局部响应应用 ReLU
        pooled = activated.amax(dim=(2, 3))  # 用全局最大池化保留最强方向响应
        logits = pooled @ self.classifier + self.bias  # 计算两类分类 logits
        return logits, convolution, pooled  # 返回分类输出和可解释中间张量
def cross_entropy(logits, target):  # 手写数值稳定多分类交叉熵
    shifted = logits - logits.max(dim=1, keepdim=True).values  # 平移每行 logits 防止指数溢出
    log_normalizer = torch.log(torch.exp(shifted).sum(dim=1))  # 计算 log-sum-exp 归一化项
    selected = shifted[torch.arange(len(target)), target]  # 读取真实类别 logit
    return (log_normalizer - selected).mean()  # 返回平均分类损失
model = TinyManualCNN()  # 创建待训练手写 CNN
training_trace = []  # 保存关键步损失和梯度
for step in range(1, 251):  # 在八张训练图上执行两百五十步
    logits, convolution, pooled = model(images[train_indices])  # 真实执行手写卷积 forward
    loss = cross_entropy(logits, labels[train_indices])  # 计算训练分类损失
    loss.backward()  # 真实执行 backward 更新卷积核和分类头
    gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in model.parameters()))  # 计算模型全局梯度范数
    with torch.no_grad():  # 关闭手写 SGD 更新计算图
        for parameter in model.parameters():  # 遍历卷积核、分类权重和偏置
            parameter -= 0.12 * parameter.grad  # 使用固定学习率更新参数
            parameter.grad.zero_()  # 清空本步梯度
    if step in {1, 5, 40, 250}:  # 保存关键训练节点
        training_trace.append((step, float(loss), float(gradient_norm)))  # 记录真实优化过程
print("step | loss | grad_norm")  # 输出 CNN 训练轨迹表头
for item in training_trace:  # 遍历四个关键节点
    print(f"{item[0]:3d} | {item[1]:.6f} | {item[2]:.6f}")  # 展示损失和梯度变化
print("中间张量：convolution", tuple(convolution.shape), "pooled", tuple(pooled.shape))  # 展示手写卷积输出形状
print("首训练图 pooled：", pooled[0].detach().tolist())  # 展示四个方向/纹理响应

step | loss | grad_norm
  1 | 0.727561 | 0.200773
  5 | 0.710332 | 0.221847
 40 | 0.477223 | 0.435820
250 | 0.005206 | 0.020633
中间张量：convolution (8, 4, 6, 6) pooled (8, 4)
首训练图 pooled： [0.9812277555465698, 1.7727347612380981, 0.9702835083007812, 2.3120548725128174]


## 十二图逐样本结果与验证指标

In [4]:
with torch.no_grad():  # 进入全量图像推理阶段
    all_logits, all_convolution, all_pooled = model(images)  # 对十二张图执行同一规范化 forward
    predictions = all_logits.argmax(dim=1)  # 获取两类预测
all_accuracy = float((predictions == labels).float().mean())  # 计算全部教学图准确率
validation_accuracy = float((predictions[validation_indices] == labels[validation_indices]).float().mean())  # 计算四张未见位置验证准确率
print("id | split | label | prediction | logits | correct")  # 输出逐图结果表头
for index in range(12):  # 遍历十二张图像
    split = "train" if index in train_indices.tolist() else "valid"  # 获取当前数据划分
    print(f"IMG-{index + 1:02d} | {split:5} | {class_names[int(labels[index])]} | {class_names[int(predictions[index])]} | {[round(value, 3) for value in all_logits[index].tolist()]} | {bool(predictions[index] == labels[index])}")  # 展示真实分类结果
print(f"accuracy：baseline={baseline_accuracy:.1%}，CNN all={all_accuracy:.1%}，validation={validation_accuracy:.1%}")  # 汇总同数据和留出位置指标

id | split | label | prediction | logits | correct
IMG-01 | train | 竖直裂纹 | 竖直裂纹 | [2.475, -2.886] | True
IMG-02 | train | 竖直裂纹 | 竖直裂纹 | [2.475, -2.886] | True
IMG-03 | train | 竖直裂纹 | 竖直裂纹 | [2.475, -2.886] | True
IMG-04 | train | 竖直裂纹 | 竖直裂纹 | [2.475, -2.886] | True
IMG-05 | valid | 竖直裂纹 | 竖直裂纹 | [2.475, -2.886] | True
IMG-06 | valid | 竖直裂纹 | 竖直裂纹 | [2.475, -2.886] | True
IMG-07 | train | 水平划痕 | 水平划痕 | [-2.731, 2.448] | True
IMG-08 | train | 水平划痕 | 水平划痕 | [-2.728, 2.446] | True
IMG-09 | train | 水平划痕 | 水平划痕 | [-2.724, 2.44] | True
IMG-10 | train | 水平划痕 | 水平划痕 | [-2.724, 2.44] | True
IMG-11 | valid | 水平划痕 | 水平划痕 | [-2.731, 2.448] | True
IMG-12 | valid | 水平划痕 | 水平划痕 | [-2.724, 2.44] | True
accuracy：baseline=50.0%，CNN all=100.0%，validation=100.0%


## 失败案例与修正：uint8 数值未除以 255

同一模型和标签下比较原始 0–255 浮点输入与规范化输入。未缩放会显著放大卷积响应、loss 或梯度，使学习率和混合精度失去预期尺度。

In [5]:
def probe_gradient(batch):  # 在模型副本上测量一个 batch 的损失和梯度尺度
    probe = TinyManualCNN()  # 创建独立探针模型
    probe.load_state_dict(model.state_dict())  # 恢复完全相同训练后参数
    logits, convolution, pooled = probe(batch)  # 对指定数值范围执行 forward
    loss = cross_entropy(logits, labels[train_indices])  # 计算相同八标签损失
    loss.backward()  # 真实执行 backward 获取尺度影响
    gradient_norm = float(torch.sqrt(sum(parameter.grad.square().sum() for parameter in probe.parameters())))  # 计算全局梯度范数
    activation_max = float(convolution.abs().max())  # 记录卷积响应最大幅值
    return float(loss), gradient_norm, activation_max  # 返回损失、梯度和激活尺度
raw_loss, raw_gradient, raw_activation = probe_gradient(images_uint8[train_indices].float())  # 复现未除以 255 的错误输入
scaled_loss, scaled_gradient, scaled_activation = probe_gradient(images[train_indices])  # 执行零到一规范化修正
print(f"raw 0-255：loss={raw_loss:.4f}，grad={raw_gradient:.4f}，activation_max={raw_activation:.2f}")  # 展示错误数值尺度
print(f"scaled 0-1：loss={scaled_loss:.4f}，grad={scaled_gradient:.4f}，activation_max={scaled_activation:.2f}")  # 展示预期输入契约
print(f"激活放大倍率={raw_activation / scaled_activation:.1f}x")  # 验证 uint8 数值约放大 255 倍

raw 0-255：loss=0.0000，grad=0.0000，activation_max=743.53
scaled 0-1：loss=0.0052，grad=0.0193，activation_max=2.92
激活放大倍率=255.0x


## 结果解读

平均亮度无法编码方向，手写卷积核在不同位置共享参数，并通过最大池化识别竖/横结构。验证图位置未在训练中出现，仍能依靠共享核分类。dtype 反例说明数据合同不是预处理细节，而是模型数值稳定性的前提。

## 生产边界

教学图只有单通道 8×8 和两类，没有相机域偏移、类别不均衡或真实增强。生产需按设备/批次 group split，保存归一化统计和类别映射，监控混淆矩阵、校准与置信度漂移。部署前还要验证 ONNX/TensorRT 前后处理一致性和真实延迟。

## 最小回归测试

In [6]:
assert len(images_uint8) >= 6 and images_uint8.dtype == torch.uint8  # 保证案例包含真实 uint8 小图
assert training_trace[-1][1] < training_trace[0][1]  # 保证实际 forward/backward 降低分类损失
assert validation_accuracy >= 0.75  # 保证模型能识别未见位置的方向结构
assert all_accuracy > baseline_accuracy  # 保证空间模型优于平均亮度基线
assert raw_activation > scaled_activation * 250.0  # 保证未缩放 uint8 激活放大真实复现
assert torch.isfinite(all_logits).all()  # 保证正确输入下全部预测数值有限